## Simulación interactiva del límite asintótico para notación $O$

La notación $O$ se utiliza para mostrar que $C(n)$ crece **al mismo ritmo o más lento** que una función de referencia $g(n)$. En otras palabras, para valores suficientemente grandes de $n$, la función $g(n)$ permite construir una cota superior para $C(n)$, ignorando constantes multiplicativas y términos de menor orden.

Formalmente, se analiza el cociente entre ambas funciones mediante el límite superior:

$$
k=\limsup_{n\to\infty}\left(\frac{C(n)}{g(n)}\right), \qquad k\in\mathbb{R}_0^+
$$

Si este valor es finito, entonces existe al menos una constante $c$ que satisface la desigualdad:

$$
C(n)\le c\cdot g(n)
$$

Cuando el cociente converge a una constante positiva $k$, una elección válida para la constante debe cumplir:

$$
c\ge k
$$

Finalmente, para un valor concreto de $c$, el punto $n_0$ se determina identificando el primer entero a partir del cual la desigualdad se cumple para todo valor posterior:

$$
n_0=\min\left\{\lceil A\rceil\in\mathbb{N}\mid \forall n\ge A,\; C(n)\le c\cdot g(n)\right\}
$$

La simulación permite modificar $C(n)$, $g(n)$, el intervalo visible, la constante $c$, los términos de menor orden y la escala de la gráfica para observar cuándo una función pertenece o no pertenece a $O(g(n))$.


In [ ]:
#@title Simulación interactiva de notación O { display-mode: "form" }
from pathlib import Path
import importlib.util
import urllib.request

RAW_URL = "https://raw.githubusercontent.com/Notas-a-Mano-serie-de-libros/3_notas-a-mano-sobre-analisis-de-complejidad-computacional/main/capitulo3/asymptotic_animation.py"
CANDIDATES = (
    Path("../asymptotic_animation.py"),
    Path("capitulo3/asymptotic_animation.py"),
    Path("asymptotic_animation.py"),
)
module_path = next((candidate for candidate in CANDIDATES if candidate.exists()), None)
if module_path is None:
    module_path = Path("asymptotic_animation.py")
    request = urllib.request.Request(RAW_URL, headers={"Cache-Control": "no-cache"})
    module_path.write_bytes(urllib.request.urlopen(request).read())

spec = importlib.util.spec_from_file_location("cap3_asymptotic_animation", module_path)
asymptotic_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(asymptotic_module)

asymptotic_module.run_big_o_app()


## Ejemplo del libro


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
from scipy.optimize import brentq

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['savefig.edgecolor'] = 'white'

plt.rcParams['figure.dpi'] = 500
plt.rcParams['savefig.dpi'] = 500

plt.rcParams['savefig.pad_inches'] = 0.05

plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

In [ ]:
# Definir las funciones
def C(n):
    return n**3 + 2*n**2 + n + 5

def g(n):
    return n**3

# Constante c
c = 2

# Definir la función diferencia para buscar n₀
def diff(n):
    return c * g(n) - C(n)

# Calcular n₀ de forma precisa
n0 = brentq(diff, 1, 1e6)  # límite ajustado para evitar overflows
n0_ceil = math.ceil(n0)

n1 = np.linspace(1, 1e4, 5000)
n2 = np.linspace(1e4, 1e20, 5000)
n = np.concatenate((n1, n2))

# Evaluar funciones
f_values = C(n)
g_values = c * g(n)

# Crear la gráfica
fig, ax = plt.subplots(figsize=(8, 3))

# Títulos y etiquetas
label_text = f'$c \\cdot g(n) = {c}n^3$'
title_text = '$C(n)$ vs $O(g(n))$'

# Graficar funciones
ax.plot(n, f_values, label='$C(n) = n^3 + 2n^2 + n + 5$', color='#1565C0')
ax.plot(n, g_values, label=label_text, color='#B71C1C')

# Línea vertical en n₀
ax.axvline(x=n0, color='#2E7D32', linestyle='--', label=rf'$n_0 = \lceil {n0:.2f} \rceil = {n0_ceil}$')

# Área donde se cumple la cota
ax.fill_between(n, f_values, g_values, where=(f_values <= g_values), color='#B71C1C', alpha=0.3)

# Escala lineal
ax.set_xscale('linear')
ax.set_yscale('linear')

# Etiquetas y leyenda
ax.set_xlabel('Tamaño de la entrada ($n$)')
ax.set_ylabel('$O(g(n))$')
ax.set_title(title_text)
ax.legend(loc='upper left', bbox_to_anchor=(0.06, 1))
ax.grid(True)

# Mostrar y guardar
plt.show()
fig.savefig('graficas/comparacion_big_O.png', bbox_inches='tight')

In [ ]:
# Definir las funciones C(n) y g(n)
def C(n):
    return n**3 + 2*n**2 + n + 5

def g(n):
    return n**3

# Constante c
c = 2

# Rango de valores de n (ajustado para escala logarítmica)
n = np.logspace(0, 20, 1000000)  # de 10^0 a 10^2 => de 1 a 100

# Calcular los valores de las funciones
f_values = C(n)
g_values = c * g(n)

# Encontrar el primer valor de n tal que C(n) <= c * g(n)
mask = f_values <= g_values
if np.any(mask):
    n0_index = np.argmax(mask)
    n0 = n[n0_index]
else:
    n0 = None
    print("No se encontró un valor de n₀ donde C(n) <= c·g(n) en el rango dado.")

# Crear la gráfica
fig, ax = plt.subplots(figsize=(8, 3))

# Configurar la etiqueta de g(n)
if c == 1:
    label_text = '$c \\cdot g(n) = n^3$'
    title_text = '$C(n)$ vs $O(g(n))$'
else:
    label_text = f'$c \\cdot g(n) = {c}n^3$'
    title_text = '$C(n)$ vs $O(g(n))$'

# Graficar funciones
ax.plot(n, f_values, label='$C(n) = n^3 + 2n^2 + n + 5$', color='#1565C0')
ax.plot(n, g_values, label=label_text, color='#B71C1C')

# Línea vertical en n₀
if n0:
    n0_ceil = math.ceil(n0)
    ax.axvline(
        x=n0,
        color='#2E7D32',
        linestyle='--',
        label=rf'$n_0 \approx \lceil {n0:.2f} \rceil = {n0_ceil}$'
    )

    ax.fill_between(n, f_values, g_values, where=(f_values <= g_values), interpolate=True, color='#B71C1C', alpha=0.3)

# Escala log-log
ax.set_xscale('log')
ax.set_yscale('log')

# Etiquetas y título
ax.set_xlabel('Tamaño de la entrada ($n$)')
ax.set_ylabel('$O(g(n))$')
ax.set_title(title_text)
ax.legend(loc='upper left', bbox_to_anchor=(0.07, 1))
ax.grid(True, which="both", ls="--", linewidth=0.5)

# Mostrar la gráfica
plt.show()

# Guardar la gráfica como archivo PNG
fig.savefig('graficas/comparacion_big_O_2.png', bbox_inches='tight')